In [1]:
import pandas as pd
import pyarrow
import os
import geopandas as gpd
import folium
import requests
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import snowflake.connector

In [2]:
!ls -la "/home/jovyan/work/data/fhvhv_cleaned.parquet"

total 887048
drwxr-xr-x 1 root root     4096 Aug 23 18:13 .
drwxrwxrwx 1 root root     4096 Aug 29 19:12 ..
-rw-r--r-- 1 root root 27020226 Aug 23 18:11 part-00000-13944809-1f46-4aef-bfed-e6f45da35cb0-c000.snappy.parquet
-rw-r--r-- 1 root root   211104 Aug 23 18:11 .part-00000-13944809-1f46-4aef-bfed-e6f45da35cb0-c000.snappy.parquet.crc
-rw-r--r-- 1 root root 27049902 Aug 23 18:11 part-00001-13944809-1f46-4aef-bfed-e6f45da35cb0-c000.snappy.parquet
-rw-r--r-- 1 root root   211336 Aug 23 18:11 .part-00001-13944809-1f46-4aef-bfed-e6f45da35cb0-c000.snappy.parquet.crc
-rw-r--r-- 1 root root 27076777 Aug 23 18:12 part-00002-13944809-1f46-4aef-bfed-e6f45da35cb0-c000.snappy.parquet
-rw-r--r-- 1 root root   211548 Aug 23 18:12 .part-00002-13944809-1f46-4aef-bfed-e6f45da35cb0-c000.snappy.parquet.crc
-rw-r--r-- 1 root root 27067667 Aug 23 18:12 part-00003-13944809-1f46-4aef-bfed-e6f45da35cb0-c000.snappy.parquet
-rw-r--r-- 1 root root   211476 Aug 23 18:12 .part-00003-13944809-1f46-4aef-bfed-e6f45

In [3]:
!ls -la /home/jovyan/work/data/fhvhv_cleaned_feb.parquet/

total 844140
drwxr-xr-x 1 root root     4096 Aug 29 19:13 .
drwxrwxrwx 1 root root     4096 Aug 29 19:12 ..
-rw-r--r-- 1 root root 25619181 Aug 29 19:12 part-00000-6b18eb12-064c-4867-8b52-4de43bd29ab3-c000.snappy.parquet
-rw-r--r-- 1 root root   200160 Aug 29 19:12 .part-00000-6b18eb12-064c-4867-8b52-4de43bd29ab3-c000.snappy.parquet.crc
-rw-r--r-- 1 root root 25738202 Aug 29 19:12 part-00001-6b18eb12-064c-4867-8b52-4de43bd29ab3-c000.snappy.parquet
-rw-r--r-- 1 root root   201088 Aug 29 19:12 .part-00001-6b18eb12-064c-4867-8b52-4de43bd29ab3-c000.snappy.parquet.crc
-rw-r--r-- 1 root root 25729864 Aug 29 19:12 part-00002-6b18eb12-064c-4867-8b52-4de43bd29ab3-c000.snappy.parquet
-rw-r--r-- 1 root root   201024 Aug 29 19:12 .part-00002-6b18eb12-064c-4867-8b52-4de43bd29ab3-c000.snappy.parquet.crc
-rw-r--r-- 1 root root 25735365 Aug 29 19:12 part-00003-6b18eb12-064c-4867-8b52-4de43bd29ab3-c000.snappy.parquet
-rw-r--r-- 1 root root   201068 Aug 29 19:12 .part-00003-6b18eb12-064c-4867-8b52-4de43

### Connect to Snowflake and Load January Data

In [4]:
load_dotenv()
conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse="compute_wh",
    database="nyc_taxi",
    schema="surge_predictor"
)

In [5]:
cs = conn.cursor()

In [6]:
cs.execute("CREATE DATABASE IF NOT EXISTS nyc_taxi")
cs.execute("CREATE SCHEMA IF NOT EXISTS surge_predictor")
cs.execute("CREATE OR REPLACE STAGE fhvhv_stage")

In [7]:
cs.execute("""
CREATE WAREHOUSE IF NOT EXISTS compute_wh
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
""")

In [8]:
cs.execute("USE DATABASE nyc_taxi")
cs.execute("USE SCHEMA surge_predictor")
cs.execute("USE WAREHOUSE compute_wh")

In [9]:
cs.execute("""
PUT file:///home/jovyan/work/data/fhvhv_cleaned.parquet/part-*.parquet
@fhvhv_stage AUTO_COMPRESS=FALSE
""")

In [10]:
cs.execute("""
CREATE OR REPLACE FILE FORMAT my_parquet_format TYPE = PARQUET
""")

cs.execute("""
CREATE OR REPLACE TABLE nyc_taxi.surge_predictor.fhvhv_trips
USING TEMPLATE (
    SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
    FROM TABLE(
        INFER_SCHEMA(
            LOCATION => '@fhvhv_stage',
            FILE_FORMAT => 'my_parquet_format'
        )
    )
)
""")

cs.execute("""
COPY INTO nyc_taxi.surge_predictor.fhvhv_trips
FROM @fhvhv_stage
FILE_FORMAT = (FORMAT_NAME = 'my_parquet_format')
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
""")

cs.execute("SELECT COUNT(*) FROM nyc_taxi.surge_predictor.fhvhv_trips")
print(cs.fetchone())

(20835763,)


### Create dbt profile

In [11]:
import os
os.makedirs(os.path.expanduser("~/.dbt"), exist_ok=True)

# Note: this writes dbt's profiles.yml to reference environment variables
# (via dbt's own {{ env_var(...) }} syntax) rather than embedding real
# credentials in the file -- safe to keep in a notebook that gets committed.
profiles_content = """
nyc_surge_predictor:
  target: dev
  outputs:
    dev:
      type: snowflake
      account: "{{ env_var('SNOWFLAKE_ACCOUNT') }}"
      user: "{{ env_var('SNOWFLAKE_USER') }}"
      password: "{{ env_var('SNOWFLAKE_PASSWORD') }}"
      role: ACCOUNTADMIN
      database: nyc_taxi
      warehouse: compute_wh
      schema: surge_predictor
      threads: 4
"""

with open(os.path.expanduser("~/.dbt/profiles.yml"), "w") as f:
    f.write(profiles_content)

print("done")

done


In [12]:
cs.execute("SHOW TABLES LIKE 'fhvhv_trips' IN SCHEMA nyc_taxi.surge_predictor")
print(cs.fetchall())

[(datetime.datetime(2026, 8, 29, 12, 17, 33, 389000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'FHVHV_TRIPS', 'NYC_TAXI', 'SURGE_PREDICTOR', 'TABLE', '', '', 20835763, 857965568, 'ACCOUNTADMIN', '1', 'OFF', 'OFF', 'OFF', None, None, 'N', 'N', 'ROLE', 'N', 'N', 'N', 'N', 'N', 'N', 'OFF', 'OFF')]


In [13]:
cs.execute("SHOW VIEWS IN SCHEMA nyc_taxi.surge_predictor")
print(cs.fetchall())

[(datetime.datetime(2026, 8, 7, 15, 30, 8, 104000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'int_hourly_zone_demand', '', 'NYC_TAXI', 'SURGE_PREDICTOR', 'ACCOUNTADMIN', '', 'create or replace   view nyc_taxi.surge_predictor."int_hourly_zone_demand"\n  \n  \n  \n  \n  as (\n    \n\nselect\n    pickup_location_id as zone_id,\n    date_trunc(\'hour\', pickup_datetime) as pickup_hour_ts,\n    count(*) as pickup_count\nfrom nyc_taxi.surge_predictor."stg_fhvhv_trips"\ngroup by 1, 2\n  )\n/* {"app": "dbt", "dbt_version": "1.11.12", "profile_name": "nyc_surge_predictor", "target_name": "dev", "node_id": "model.nyc_surge_predictor.int_hourly_zone_demand"} */;', 'false', 'false', 'ROLE', 'OFF'), (datetime.datetime(2026, 8, 7, 15, 30, 6, 47000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'stg_fhvhv_trips', '', 'NYC_TAXI', 'SURGE_PREDICTOR', 'ACCOUNTADMIN', '', 'create or replace   view nyc_taxi.surge_predictor."stg_fhvhv_trips"\n  \n  \n  \n  \n  a

In [14]:
cs.execute("DESCRIBE TABLE nyc_taxi.surge_predictor.fhvhv_trips")
for row in cs.fetchall():
    print(row[0], row[1])

PU_Zone VARCHAR(16777216)
PU_Borough VARCHAR(16777216)
DO_Zone VARCHAR(16777216)
DO_Borough VARCHAR(16777216)
trip_miles FLOAT
trip_time NUMBER(38,0)
base_passenger_fare FLOAT
tolls FLOAT
access_a_ride_flag VARCHAR(16777216)
wav_request_flag VARCHAR(16777216)
wav_match_flag VARCHAR(16777216)
cbd_congestion_fee FLOAT
bcf FLOAT
sales_tax FLOAT
congestion_surcharge FLOAT
airport_fee FLOAT
request_datetime NUMBER(38,0)
on_scene_datetime NUMBER(38,0)
pickup_datetime NUMBER(38,0)
dropoff_datetime NUMBER(38,0)
pickup_hour NUMBER(38,0)
day_of_week NUMBER(38,0)
is_weekend NUMBER(38,0)
pickup_date DATE
DOLocationID NUMBER(38,0)
PULocationID NUMBER(38,0)
hvfhs_license_num VARCHAR(16777216)
dispatching_base_num VARCHAR(16777216)
originating_base_num VARCHAR(16777216)
tips FLOAT
driver_pay FLOAT
shared_request_flag VARCHAR(16777216)
shared_match_flag VARCHAR(16777216)


In [15]:
cs.execute('SELECT "pickup_datetime" FROM nyc_taxi.surge_predictor.fhvhv_trips LIMIT 5')
print(cs.fetchall())

[(1767227178000000,), (1767226435000000,), (1767227441000000,), (1767226365000000,), (1767228947000000,)]


In [16]:
cs.execute('SELECT DISTINCT "hvfhs_license_num" FROM nyc_taxi.surge_predictor.fhvhv_trips')
print(cs.fetchall())

[('HV0003',), ('HV0005',)]


### Insert Feb Data into Snowflake

In [17]:
cs.execute("USE DATABASE nyc_taxi")
cs.execute("USE SCHEMA surge_predictor")
cs.execute("USE WAREHOUSE compute_wh")

cs.execute("CREATE OR REPLACE STAGE fhvhv_stage_feb")

cs.execute("""
PUT file:///home/jovyan/work/data/fhvhv_cleaned_feb.parquet/part-*.parquet
@fhvhv_stage_feb AUTO_COMPRESS=FALSE
""")
# Append. This adds to the existing table.
cs.execute("""
COPY INTO nyc_taxi.surge_predictor.fhvhv_trips
FROM @fhvhv_stage_feb
FILE_FORMAT = (FORMAT_NAME = 'my_parquet_format')
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
""")


cs.execute("SELECT COUNT(*) FROM nyc_taxi.surge_predictor.fhvhv_trips")
print(cs.fetchone())

(40702895,)


In [19]:
cs.execute("""
SELECT
    DATE_TRUNC('month', pickup_datetime) AS month,
    COUNT(*) AS row_count
FROM nyc_taxi.surge_predictor.fhvhv_trips
GROUP BY 1
ORDER BY 1;
""")
for row in cs.fetchall():
    print(row)

ProgrammingError: 000904 (42000): SQL compilation error: error line 2 at position 24
invalid identifier 'PICKUP_DATETIME'

In [20]:
cs.execute("TRUNCATE TABLE nyc_taxi.surge_predictor.fhvhv_trips")
cs.execute("SELECT COUNT(*) FROM nyc_taxi.surge_predictor.fhvhv_trips")
print(cs.fetchone())  # should be (0,)

(0,)
